In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd


In [2]:
BASE = Path("/workspace/geovision-cali-hf")
PATHS = {
    "metadata_s2_v5b": BASE / "outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500/metadata.jsonl",
    "embeddings_remoteclip_v5b": BASE / "outputs/clip_training_remoteclip_v10_v5b_embeddings/embeddings_remoteclip_v10_v5b.npz",
    "checkpoint_sae_v5b": BASE / "outputs/clip_training_remoteclip_fusion_ksae_v10_v5b_gsplit_seed_sweep/seed57_k12_drop25_wd1e3_best.pt",
    "grid_s3": BASE / "outputs/situacion3/02_grilla_s2_features/grid_cali_005deg.parquet",
    "station_to_grid_s3": BASE / "outputs/situacion3/02_grilla_s2_features/station_to_s2_grid_mapping.csv",
    "dagma_daily_long": BASE / "outputs/situacion3/01_panel_dagma/dagma_sisaire_daily_long.parquet",
}

In [3]:
def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()
rows = []
for name, path in PATHS.items():
    rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else None,
        "md5": md5_file(path) if path.exists() and path.is_file() else None,
    })
pd.DataFrame(rows)

,name,path,exists,size_mb,md5
0,metadata_s2_v5b,/workspace/geovision-cali-hf/outputs/clip_data...,True,2.376,5e0a96c7572e00f21edb8e7a864b977d
1,embeddings_remoteclip_v5b,/workspace/geovision-cali-hf/outputs/clip_trai...,True,2.635,6cb3b11bede28d5568b9253a9a5e1d8d
2,checkpoint_sae_v5b,/workspace/geovision-cali-hf/outputs/clip_trai...,True,9.028,2a512b70e4a837a0c652aa71f2c970b4
3,grid_s3,/workspace/geovision-cali-hf/outputs/situacion...,True,0.033,cf11e3abf87e99070475a226e363f10b
4,station_to_grid_s3,/workspace/geovision-cali-hf/outputs/situacion...,True,0.001,089f931167b57a9e1d074996844e1cbd
5,dagma_daily_long,/workspace/geovision-cali-hf/outputs/situacion...,True,0.515,581644c3b09f1aacc148388bf8f62e2f


In [4]:
meta = pd.read_json(PATHS["metadata_s2_v5b"], lines=True)
emb_npz = np.load(PATHS["embeddings_remoteclip_v5b"], allow_pickle=True)
print("Metadata shape:", meta.shape)
print("Metadata columns:")
print(meta.columns.tolist())
print("\nNPZ keys:", emb_npz.files)
for key in emb_npz.files:
    arr = emb_npz[key]
    print(key, arr.shape, arr.dtype)
emb = emb_npz["remoteclip_visual_512"]
pair_ids_emb = emb_npz["pair_id"].astype(str)
pair_ids_meta = meta["pair_id"].astype(str).to_numpy()
alignment_ok = np.array_equal(pair_ids_emb, pair_ids_meta)
summary = {
    "n_metadata": len(meta),
    "n_embeddings": emb.shape[0],
    "embedding_dim": emb.shape[1],
    "pair_id_alignment_ok": alignment_ok,
    "date_min": str(pd.to_datetime(meta["date_day"]).min().date()),
    "date_max": str(pd.to_datetime(meta["date_day"]).max().date()),
    "n_dates": int(meta["date_day"].nunique()),
    "n_scenes": int(meta["scene_id"].nunique()),
    "n_mgrs_tiles": int(meta["mgrs_tile"].nunique()),
    "image_shapes": meta["image_shape"].astype(str).value_counts().to_dict(),
}
summary

Metadata shape: (1500, 38)
Metadata columns:
['pair_id', 'split', 'candidate_class', 'pdf_class', 'candidate_class_original', 'text', 'date_day', 'scene_id', 'mgrs_tile', 'row_off', 'col_off', 'image_path', 'scl_path', 'image_shape', 'scl_shape', 'image_dtype', 'scl_dtype', 's2_bands', 's5p_used_as', 'scl_used_as', 'created_at_utc', 'image_min', 'image_max', 'image_mean', 'image_std', 'ndvi_mean', 'ndbi_mean', 'ndwi_mean', 'scl_cloud_shadow_pct', 'scl_valid_visual_pct', 'zero_pct', 'high_reflectance_pct', 's5p_lat_center', 's5p_lon_center', 's5p_no2_column', 's5p_so2_column', 's5p_o3_column', 'scl_unique_values']

NPZ keys: ['remoteclip_visual_512', 'pair_id', 'image_path']
remoteclip_visual_512 (1500, 512) float32
pair_id (1500,) object
image_path (1500,) object


{'n_metadata': 1500,
 'n_embeddings': 1500,
 'embedding_dim': 512,
 'pair_id_alignment_ok': True,
 'date_min': '2020-01-02',
 'date_max': '2024-12-16',
 'n_dates': 115,
 'n_scenes': 115,
 'n_mgrs_tiles': 2,
 'image_shapes': {'[64, 64, 12]': 1500}}

In [5]:
import torch
ckpt = torch.load(PATHS["checkpoint_sae_v5b"], map_location="cpu")
print("Checkpoint keys:", ckpt.keys())
for key in ["classes", "config", "dataset", "embedding_npz"]:
    if key in ckpt:
        print(f"\n{key}:")
        print(ckpt[key])
state = ckpt["model_state_dict"]
print("\nPrimeras 30 claves del state_dict:")
for i, k in enumerate(state.keys()):
    if i >= 30:
        break
    print(i, k, tuple(state[k].shape) if hasattr(state[k], "shape") else type(state[k]))
shape_summary = []
for k, v in state.items():
    if hasattr(v, "shape"):
        shape_summary.append({
            "key": k,
            "shape": tuple(v.shape),
        })
pd.DataFrame(shape_summary)

Checkpoint keys: dict_keys(['model_state_dict', 'classes', 'config', 'dataset', 'embedding_npz'])

classes:
['contaminacion_alta_NO2', 'contaminacion_alta_SO2', 'ozono_anomalo', 'suelo_urbano', 'vegetacion_densa']

config:
{'name': 'seed57_k12_drop25_wd1e3', 'k_frac': 0.12, 'l1': 0.004, 'alpha': 0.1, 'lr': 0.00015, 'wd': 0.001, 'drop': 0.25, 'epochs': 90, 'batch': 64, 'seed_offset': 57}

dataset:
/workspace/geovision-cali-hf/outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500_gsplit

embedding_npz:
/workspace/geovision-cali-hf/outputs/clip_training_remoteclip_v10_v5b_embeddings/embeddings_remoteclip_v10_v5b.npz

Primeras 30 claves del state_dict:
0 logit_scale ()
1 img.0.weight (595,)
2 img.0.bias (595,)
3 img.1.weight (768, 595)
4 img.1.bias (768,)
5 img.4.weight (512, 768)
6 img.4.bias (512,)
7 img.7.weight (256, 512)
8 img.7.bias (256,)
9 txt.0.weight (384,)
10 txt.0.bias (384,)
11 txt.1.weight (512, 384)
12 txt.1.bias (512,)
13 txt.3.weight (256,

/tmp/ipykernel_942700/1367501780.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(PATHS["checkpoint_sae_v5b"], map_location="cpu")


,key,shape
0,logit_scale,()
1,img.0.weight,"(595,)"
2,img.0.bias,"(595,)"
3,img.1.weight,"(768, 595)"
4,img.1.bias,"(768,)"
5,img.4.weight,"(512, 768)"
6,img.4.bias,"(512,)"
7,img.7.weight,"(256, 512)"
8,img.7.bias,"(256,)"
9,txt.0.weight,"(384,)"


In [6]:
def aux_row(r):
    img = np.load(r.image_path).astype("float32")

    red = img[:, :, 3]
    green = img[:, :, 2]
    nir = img[:, :, 7]
    swir = img[:, :, 10]

    ndvi = (nir - red) / (nir + red + 1e-6)
    ndbi = (swir - nir) / (swir + nir + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)

    arr = img.reshape(-1, 12)

    vals = []
    vals.extend(arr.mean(0))
    vals.extend(arr.std(0))
    vals.extend(np.percentile(arr, [10, 50, 90], axis=0).ravel())

    for idx in [ndvi, ndbi, ndwi]:
        vals.extend([
            np.nanmean(idx),
            np.nanstd(idx),
            np.nanpercentile(idx, 10),
            np.nanpercentile(idx, 50),
            np.nanpercentile(idx, 90),
        ])

    d = pd.to_datetime(r.date_day)
    doy = d.dayofyear

    vals.extend([
        float(d.year),
        np.sin(2 * np.pi * doy / 366),
        np.cos(2 * np.pi * doy / 366),
        float(r.ndvi_mean),
        float(r.ndbi_mean),
        float(r.ndwi_mean),
        float(r.scl_cloud_shadow_pct),
        float(r.scl_valid_visual_pct),
    ])

    return np.array(vals, dtype="float32")


aux = np.stack([aux_row(r) for _, r in meta.iterrows()])
remote = emb.astype("float32")

aux_summary = {
    "remote_shape": tuple(remote.shape),
    "aux_shape": tuple(aux.shape),
    "combined_input_dim": int(remote.shape[1] + aux.shape[1]),
    "expected_checkpoint_input_dim": int(state["img.0.weight"].shape[0]),
    "matches_checkpoint_input": int(remote.shape[1] + aux.shape[1]) == int(state["img.0.weight"].shape[0]),
    "aux_nan_count": int(np.isnan(aux).sum()),
    "remote_nan_count": int(np.isnan(remote).sum()),
}
aux_summary


{'remote_shape': (1500, 512),
 'aux_shape': (1500, 83),
 'combined_input_dim': 595,
 'expected_checkpoint_input_dim': 595,
 'matches_checkpoint_input': True,
 'aux_nan_count': 0,
 'remote_nan_count': 0}

In [7]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IN = 595
EMB = 256
SAE_H = 1024
k_frac = ckpt["config"]["k_frac"]
drop = ckpt["config"]["drop"]

class SAE(nn.Module):
    def __init__(self, k_frac=0.15):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(EMB, SAE_H), nn.ReLU())
        self.dec = nn.Linear(SAE_H, EMB)
        self.k_frac = k_frac

    def forward(self, x):
        z = self.enc(x)
        k = max(1, int(z.shape[1] * self.k_frac))
        vals, idx = torch.topk(z, k, dim=1)
        sparse = torch.zeros_like(z).scatter(1, idx, vals)
        return sparse, self.dec(sparse)

class VisualProjectorSAE(nn.Module):
    def __init__(self, drop=0.25, k_frac=0.12):
        super().__init__()
        self.img = nn.Sequential(
            nn.LayerNorm(IN),
            nn.Linear(IN, 768),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(512, EMB),
        )
        self.sae_i = SAE(k_frac)

    def forward_img(self, x):
        h = self.img(x)
        z, r = self.sae_i(h)
        return h, z, r

model = VisualProjectorSAE(drop=drop, k_frac=k_frac).to(DEVICE)
visual_state = {
    k: v for k, v in ckpt["model_state_dict"].items()
    if k.startswith("img.") or k.startswith("sae_i.")
}
missing, unexpected = model.load_state_dict(visual_state, strict=False)
model.eval()

train_mask = meta["split"].to_numpy() == "train"
sc_remote = StandardScaler().fit(remote[train_mask])
sc_aux = StandardScaler().fit(aux[train_mask])

X_595 = np.concatenate([
    sc_remote.transform(remote),
    sc_aux.transform(aux),
], axis=1).astype("float32")

with torch.no_grad():
    x_t = torch.tensor(X_595, dtype=torch.float32, device=DEVICE)
    h_256, z_1024, r_256 = model.forward_img(x_t)

h_256 = h_256.detach().cpu().numpy().astype("float32")
z_1024 = z_1024.detach().cpu().numpy().astype("float32")
r_256 = r_256.detach().cpu().numpy().astype("float32")

embedding256_summary = {
    "device": str(DEVICE),
    "missing_keys": missing,
    "unexpected_keys": unexpected,
    "X_595_shape": tuple(X_595.shape),
    "h_256_shape": tuple(h_256.shape),
    "z_1024_shape": tuple(z_1024.shape),
    "r_256_shape": tuple(r_256.shape),
    "h_256_nan_count": int(np.isnan(h_256).sum()),
    "r_256_nan_count": int(np.isnan(r_256).sum()),
    "z_1024_sparsity_ratio": float((np.abs(z_1024) < 1e-8).mean()),
}

embedding256_summary


{'device': 'cuda',
 'missing_keys': [],
 'unexpected_keys': [],
 'X_595_shape': (1500, 595),
 'h_256_shape': (1500, 256),
 'z_1024_shape': (1500, 1024),
 'r_256_shape': (1500, 256),
 'h_256_nan_count': 0,
 'r_256_nan_count': 0,
 'z_1024_sparsity_ratio': 0.880859375}

In [8]:
grid = pd.read_parquet(PATHS["grid_s3"]).copy()

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

tile_map_rows = []

grid_lat = grid["lat"].to_numpy()
grid_lon = grid["lon"].to_numpy()
grid_ids = grid["grid_id"].to_numpy()

for i, row in meta.reset_index(drop=True).iterrows():
    lat = float(row["s5p_lat_center"])
    lon = float(row["s5p_lon_center"])

    dists = haversine_km(lon, lat, grid_lon, grid_lat)
    j = int(np.argmin(dists))

    tile_map_rows.append({
        "pair_id": row["pair_id"],
        "date_day": row["date_day"],
        "scene_id": row["scene_id"],
        "mgrs_tile": row["mgrs_tile"],
        "image_path": row["image_path"],
        "scl_path": row["scl_path"],
        "lat": lat,
        "lon": lon,
        "nearest_grid_id": grid_ids[j],
        "grid_lat": float(grid_lat[j]),
        "grid_lon": float(grid_lon[j]),
        "distance_km_to_grid": float(dists[j]),
        "split_s2": row["split"],
        "candidate_class": row["candidate_class"],
    })

tile_to_grid = pd.DataFrame(tile_map_rows)

coverage_summary = {
    "n_tiles": int(len(tile_to_grid)),
    "n_unique_grid_cells": int(tile_to_grid["nearest_grid_id"].nunique()),
    "n_unique_dates": int(tile_to_grid["date_day"].nunique()),
    "date_min": str(pd.to_datetime(tile_to_grid["date_day"]).min().date()),
    "date_max": str(pd.to_datetime(tile_to_grid["date_day"]).max().date()),
    "max_distance_km_to_grid": float(tile_to_grid["distance_km_to_grid"].max()),
    "median_distance_km_to_grid": float(tile_to_grid["distance_km_to_grid"].median()),
    "cells_with_at_least_1_tile": int(tile_to_grid.groupby("nearest_grid_id").size().ge(1).sum()),
    "cells_with_at_least_8_dates": int(tile_to_grid.groupby("nearest_grid_id")["date_day"].nunique().ge(8).sum()),
    "max_dates_per_cell": int(tile_to_grid.groupby("nearest_grid_id")["date_day"].nunique().max()),
    "median_dates_per_cell": float(tile_to_grid.groupby("nearest_grid_id")["date_day"].nunique().median()),
}

coverage_summary


{'n_tiles': 1500,
 'n_unique_grid_cells': 594,
 'n_unique_dates': 115,
 'date_min': '2020-01-02',
 'date_max': '2024-12-16',
 'max_distance_km_to_grid': 0.33413096003202375,
 'median_distance_km_to_grid': 0.21299660111796487,
 'cells_with_at_least_1_tile': 594,
 'cells_with_at_least_8_dates': 1,
 'max_dates_per_cell': 9,
 'median_dates_per_cell': 2.0}